# Phase 8c — does a reranker beat RRF?

The Phase 8 failure analysis found several eval questions where the correct article *was* retrieved but sitting outside the top-5 (q25, q29, q31 at rank 7). Raising `top_k` to 10 (Phase 8b) fixed those. This notebook asks a different question: would a **reranker** get the correct article ranked *higher* in the first place, rather than just widening the window?

**Two pipelines compared, both starting from the same raw candidates**:
- **RRF (current production)**: `Retriever.retrieve()` — rank-fuses Pinecone semantic top-50 + BM25 keyword top-50 using reciprocal rank, no document text is read a second time.
- **Reranker**: pool the semantic + keyword candidates (deduped, no RRF math), send the pooled documents + query to Cohere `rerank-v3.5` (~$0.002/query, multilingual, 4096-token context), and rank by its output relevance score instead.

Scored the same way as every other retrieval decision in this project (`scripts/05. search_method_evaluation.ipynb`, `scripts/06. retrieval.ipynb`): recall@1/5/10 + MRR over the 40 eval questions, split by same-language vs cross-lingual. This is a retrieval-accuracy comparison only (cheap, ~40 rerank calls) — if it looks promising, a follow-up can extend it through generation + LLM-judge the way Phase 8b did for `top_k`/`temperature`.

Requires `COHERE_API_KEY`, `PINECONE_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever

retriever = Retriever()

EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

LAW_MAP = {
    "Work_Law_1996": "labor96",
    "Social_Security_Law": "ssl",
    "Professional_Work_Law_2019": "prof19",
}
eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]
print(f"{len(eval_questions)} eval questions loaded")

40 eval questions loaded


## Reranked retrieval

Pool = dedup(semantic top-30 ids, keyword top-30 ids) — 30 each keeps the pool comfortably under Cohere's 100-document-per-search-unit limit even after the union, so this stays 1 billed search per query. Cohere reranks by document *text*, so chunk texts are pulled straight from `retriever.chunks`.

In [2]:
import time

RERANK_MODEL = "rerank-v3.5"
POOL_SIZE = 30
MAX_RETRIES = 6

def retrieve_reranked(query, top_k=10, pool_size=POOL_SIZE):
    semantic_ids = retriever._semantic_candidates(query, pool_size)
    keyword_ids = retriever._keyword_candidates(query, pool_size)
    pool_ids = list(dict.fromkeys(semantic_ids + keyword_ids))  # dedup, order doesn't matter pre-rerank

    rows = [retriever.chunks.iloc[retriever.id_to_row[cid]] for cid in pool_ids]
    documents = [r["text"] for r in rows]

    resp = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = retriever.cohere_client.rerank(model=RERANK_MODEL, query=query, documents=documents, top_n=top_k)
            break
        except Exception as e:
            wait = min(60, 5 * (2 ** attempt))
            print(f"  cohere rerank retry {attempt + 1}: {type(e).__name__} -- waiting {wait}s")
            time.sleep(wait)
    if resp is None:
        raise RuntimeError("Cohere rerank failed after retries")

    results = []
    for res in resp.results:
        r = rows[res.index]
        results.append({
            "chunk_id": pool_ids[res.index],
            "score": res.relevance_score,
            "law": r["law"], "language": r["language"],
            "article_number": r["article_number"], "clause": r["clause"], "text": r["text"],
        })
    return results

# sanity check on one question
demo = retrieve_reranked(eval_questions.iloc[2]["question_text"], top_k=5)
for d in demo:
    print(d["law"], d["article_number"], d["language"], round(d["score"], 4))

labor96 66 en 0.3103
labor96 66 ar 0.3046
labor96 61 ar 0.2666
labor96 65 en 0.265
labor96 61 en 0.231


## Head-to-head: RRF vs reranker, recall@1/5/10 + MRR

In [3]:
def rank_of_correct(results, row):
    correct_articles = set(row.answer_article_numbers)
    for pos, r in enumerate(results, start=1):
        if r["law"] == row.answer_law_code and r["language"] == row.answer_language and r["article_number"] in correct_articles:
            return pos
    return None

rows = []
for row in eval_questions.itertuples(index=False):
    rrf_results = retriever.retrieve(row.question_text, top_k=10)
    rerank_results = retrieve_reranked(row.question_text, top_k=10)
    rows.append({
        "question_id": row.question_id,
        "is_cross_lingual": row.is_cross_lingual,
        "rrf_rank": rank_of_correct(rrf_results, row),
        "rerank_rank": rank_of_correct(rerank_results, row),
    })

details = pd.DataFrame(rows)

def metrics(df, col):
    n = len(df)
    r = df[col]
    return {"n": n, "recall@1": (r == 1).sum() / n, "recall@5": (r.notna() & (r <= 5)).sum() / n,
            "recall@10": r.notna().sum() / n, "mrr": (1 / r.dropna()).sum() / n}

for label, subset in [("overall", details), ("same-language", details[~details["is_cross_lingual"]]), ("cross-lingual", details[details["is_cross_lingual"]])]:
    print(f"--- {label} ---")
    print("RRF:     ", metrics(subset, "rrf_rank"))
    print("reranker:", metrics(subset, "rerank_rank"))

details.to_csv(PROJECT / "eval" / "reranker_comparison.csv", index=False, encoding="utf-8-sig")

--- overall ---
RRF:      {'n': 40, 'recall@1': np.float64(0.675), 'recall@5': np.float64(0.825), 'recall@10': np.float64(0.9), 'mrr': np.float64(0.744047619047619)}
reranker: {'n': 40, 'recall@1': np.float64(0.6), 'recall@5': np.float64(0.925), 'recall@10': np.float64(0.925), 'mrr': np.float64(0.7308333333333333)}
--- same-language ---
RRF:      {'n': 32, 'recall@1': np.float64(0.84375), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.96875), 'mrr': np.float64(0.8742559523809523)}
reranker: {'n': 32, 'recall@1': np.float64(0.71875), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.9375), 'mrr': np.float64(0.8041666666666667)}
--- cross-lingual ---
RRF:      {'n': 8, 'recall@1': np.float64(0.0), 'recall@5': np.float64(0.375), 'recall@10': np.float64(0.625), 'mrr': np.float64(0.2232142857142857)}
reranker: {'n': 8, 'recall@1': np.float64(0.125), 'recall@5': np.float64(0.875), 'recall@10': np.float64(0.875), 'mrr': np.float64(0.4375)}


## Where do they disagree?

Questions where one pipeline found the correct article and the other didn't (or ranked it meaningfully differently) — the interesting cases, not the ones where both already agree.

In [4]:
diffs = details[details["rrf_rank"] != details["rerank_rank"]].copy()
diffs = diffs.merge(eval_questions[["question_id", "question_text", "question_category"]], on="question_id")
print(f"{len(diffs)}/{len(details)} questions ranked differently")
diffs[["question_id", "question_category", "is_cross_lingual", "rrf_rank", "rerank_rank"]]

15/40 questions ranked differently


,question_id,question_category,is_cross_lingual,rrf_rank,rerank_rank
0,q01,termination,False,1.0,2.0
1,q03,rights,False,1.0,5.0
2,q04,termination,True,NaN,2.0
3,q06,rights,False,1.0,2.0
4,q07,obligations,False,1.0,3.0
5,q08,penalties,True,NaN,4.0
6,q20,penalties,False,3.0,5.0
7,q22,benefits_calculation,False,NaN,NaN
8,q23,scope,False,4.0,NaN
9,q25,definitions,True,7.0,4.0


## RRF, then rerank\n\nThe reranker-on-raw-pool approach above bypasses RRF entirely and lost same-language precision (recall@1 0.844 → 0.719, MRR 0.874 → 0.804) even though it fixed cross-lingual recall dramatically. A third option: let RRF do what it's already good at (fuse semantic + keyword into a sensible ranking) over a **wider** window than the final top-k, then let the reranker do a final precision pass on just those RRF-ranked candidates instead of the raw pool. This keeps RRF's fusion signal instead of throwing it away, while still giving the reranker a shot at fixing rank-order mistakes.\n\nRRF candidate window: top-20 (2x the final `top_k=10`) — enough headroom for the reranker to reorder without processing the full raw pool again.

In [5]:
RRF_POOL_SIZE = 20

def retrieve_rrf_then_rerank(query, top_k=10, rrf_pool_size=RRF_POOL_SIZE):
    rrf_candidates = retriever.retrieve(query, top_k=rrf_pool_size)
    documents = [c["text"] for c in rrf_candidates]

    resp = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = retriever.cohere_client.rerank(model=RERANK_MODEL, query=query, documents=documents, top_n=top_k)
            break
        except Exception as e:
            wait = min(60, 5 * (2 ** attempt))
            print(f"  cohere rerank retry {attempt + 1}: {type(e).__name__} -- waiting {wait}s")
            time.sleep(wait)
    if resp is None:
        raise RuntimeError("Cohere rerank failed after retries")

    results = []
    for res in resp.results:
        c = rrf_candidates[res.index]
        results.append({**c, "score": res.relevance_score})
    return results

# sanity check
demo = retrieve_rrf_then_rerank(eval_questions.iloc[2]["question_text"], top_k=5)
for d in demo:
    print(d["law"], d["article_number"], d["language"], round(d["score"], 4))

labor96 66 en 0.3103
labor96 66 ar 0.3046
labor96 61 ar 0.2666
labor96 65 en 0.265
labor96 61 en 0.231


## Three-way comparison: RRF vs raw-pool-rerank vs RRF-then-rerank

In [6]:
rows3 = []
for row in eval_questions.itertuples(index=False):
    rrf_then_rerank_results = retrieve_rrf_then_rerank(row.question_text, top_k=10)
    rows3.append({
        "question_id": row.question_id,
        "is_cross_lingual": row.is_cross_lingual,
        "rrf_then_rerank_rank": rank_of_correct(rrf_then_rerank_results, row),
    })

details3 = details.merge(pd.DataFrame(rows3), on=["question_id", "is_cross_lingual"])

for label, subset in [("overall", details3), ("same-language", details3[~details3["is_cross_lingual"]]), ("cross-lingual", details3[details3["is_cross_lingual"]])]:
    print(f"--- {label} ---")
    print("RRF only:        ", metrics(subset, "rrf_rank"))
    print("rerank raw pool: ", metrics(subset, "rerank_rank"))
    print("RRF then rerank: ", metrics(subset, "rrf_then_rerank_rank"))

details3.to_csv(PROJECT / "eval" / "reranker_comparison.csv", index=False, encoding="utf-8-sig")

--- overall ---
RRF only:         {'n': 40, 'recall@1': np.float64(0.675), 'recall@5': np.float64(0.825), 'recall@10': np.float64(0.9), 'mrr': np.float64(0.744047619047619)}
rerank raw pool:  {'n': 40, 'recall@1': np.float64(0.6), 'recall@5': np.float64(0.925), 'recall@10': np.float64(0.925), 'mrr': np.float64(0.7308333333333333)}
RRF then rerank:  {'n': 40, 'recall@1': np.float64(0.6), 'recall@5': np.float64(0.9), 'recall@10': np.float64(0.9), 'mrr': np.float64(0.7216666666666667)}
--- same-language ---
RRF only:         {'n': 32, 'recall@1': np.float64(0.84375), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.96875), 'mrr': np.float64(0.8742559523809523)}
rerank raw pool:  {'n': 32, 'recall@1': np.float64(0.71875), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.9375), 'mrr': np.float64(0.8041666666666667)}
RRF then rerank:  {'n': 32, 'recall@1': np.float64(0.71875), 'recall@5': np.float64(0.9375), 'recall@10': np.float64(0.9375), 'mrr': np.float64(0.8057291666666